In [ ]:
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())
token = os.getenv("KAGGLE_API_TOKEN")

# !kaggle competitions download -c traffic-sign-object-detection-challenge
# !unzip traffic-sign-object-detection-challenge.zip

In [ ]:
from utils.data import convert_coco_to_yolo, generate_yolo_yaml
from utils.inference import run_inference
from utils.viz import visualize_predictions
from utils.train import train_yolo_model
import pandas as pd

In [ ]:
infer_class_map = convert_coco_to_yolo(
    ann_path="dataset/train/annotations.json", 
    img_src_dir="dataset/train/images", 
    yolo_base_dir="dataset/obj"
)
generate_yolo_yaml("data.yaml", "dataset/obj", infer_class_map)

In [ ]:
base_cfg = {
    "data_yaml": "data.yaml",
    "epochs": 64,
    "imgsz": 768,
    "batch": 16,
    "patience": 8,
    "amp": True,
    "device": 0
}

In [ ]:
train_yolo_model(
    model_variant="yolo26n.pt", 
    experiment_name="traffic_signs_v26",
    **base_cfg
)

In [ ]:
results = run_inference(
    model_path="runs/detect/traffic_signs_v26_testing/weights/best.pt",
    test_dir="dataset/test/images",
    class_names=infer_class_map
)

In [ ]:
df = pd.DataFrame(results)
df.to_csv("submission-v26-new.csv", index=False)

visualize_predictions(results, "dataset/test/images", infer_class_map)